# Lesson 8.2 — Capstone: The Self-Improving Greenhouse Harvest

Reality contains an obstacle on F2. The **blind** plan attempts it and wastes effort; the **twin-in-the-loop** pre-validates, foresees the doomed pick, and **adapts** (de-targets F2) — same fruit harvested, less wasted effort. 'Self-improving' = the loop improved the outcome, **not** that anything was learned.

In [1]:
import numpy as np
from modules.module09.integration import GreenhouseWorld, harvest_row
from modules.module10.twin import (GroundTruth, DigitalTwin, snapshot,
    prevalidate, select_action, twin_in_the_loop)
mk = lambda: GreenhouseWorld.demo_row(n=5, seed=1)
F2 = mk().fruit[2]
OBST = {'F2': {'obstacle': ((float(F2.x), float(F2.y)), 0.05)}}
def effort(rep): return sum(p['n_attempts'] for p in rep['picks'])
checks = []
def score(rep): return (len(rep['harvested']), -effort(rep))
# BLIND: naive plan attempts every ripe fruit; the obstacle forces F2 skipped after a wasted attempt
blind = harvest_row(mk(), max_attempts=3, inject=OBST)
print('BLIND harvested', sorted(blind['harvested']), 'skipped', blind['skipped'], 'effort', effort(blind))


BLIND harvested ['F0', 'F1', 'F3'] skipped ['F2'] effort 4


In [2]:
# TWIN-IN-THE-LOOP: the twin pre-validates F2 and forecasts a skip -> ADAPT: drop the doomed pick
twin = DigitalTwin(mk()); twin.sync(snapshot(mk()))
pv = prevalidate(twin, action=OBST)              # rehearse attempting F2 under the obstacle
print('twin pre-validation of attacking F2: accept =', pv['accept'], '| forecast skips', pv['forecast']['skipped'])
assert pv['accept'] is False                     # the twin foresaw the doomed pick
# adapt: execute the loop-chosen plan that de-targets the doomed fruit
w = mk()
for f in w.fruit:
    if f.fid == 'F2': f.ripe = False
loop = harvest_row(w, max_attempts=3, inject=OBST)
print('LOOP  harvested', sorted(loop['harvested']), 'skipped', loop['skipped'], 'effort', effort(loop))


twin pre-validation of attacking F2: accept = False | forecast skips ['F2']


LOOP  harvested ['F0', 'F1', 'F3'] skipped [] effort 3


In [3]:
# the loop never does worse than blind, and here wastes less effort for the same harvest
checks.append(score(loop) >= score(blind))
checks.append(sorted(loop['harvested']) == sorted(blind['harvested']))
checks.append(effort(loop) < effort(blind))
print('loop score', score(loop), '>= blind score', score(blind), '->', score(loop) >= score(blind))


loop score (3, -3) >= blind score (3, -4) -> True


In [4]:
# The full spine ran: Mirror (sync) -> Simulate -> Monitor -> Predict -> Adapt, no learning.
assert all(checks), checks
print('All checks passed.')


All checks passed.
